# Orthostatic Stress Testing and Syncope Evaluation

**Clinical Application:** Syncope evaluation, dysautonomia diagnosis, orthostatic hypotension assessment

**Learning Objectives:**
1. Understand normal orthostatic cardiovascular responses
2. Simulate tilt table testing and autonomic compensation
3. Identify pathological orthostatic responses
4. Differentiate types of orthostatic intolerance

**Clinical Relevance:**
- Syncope evaluation (30% of population experiences syncope)
- Orthostatic hypotension (OH) diagnosis
- Postural orthostatic tachycardia syndrome (POTS)
- Autonomic failure (pure autonomic failure, MSA)
- Falls prevention in elderly

**Physiological Background:**

Standing causes gravitational pooling of ~500-1000 mL blood in lower extremities and splanchnic circulation:

**Immediate Effects (0-30 seconds):**
- ↓ Venous return → ↓ Stroke volume (40%)
- ↓ Cardiac output → ↓ Blood pressure
- Baroreceptor activation within 1-2 cardiac cycles

**Compensatory Mechanisms:**
1. **Neural** (seconds): Vagal withdrawal + sympathetic activation
   - ↑ Heart rate (10-25 bpm)
   - ↑ Cardiac contractility
   - ↑ Peripheral vasoconstriction (SVR ↑ 30%)
2. **Humoral** (minutes): RAAS activation, vasopressin release
3. **Skeletal muscle pump**: Leg muscle contractions aid venous return

**Normal Response:**
- Systolic BP: ↓ <20 mmHg or stable
- Diastolic BP: ↑ 5-10 mmHg (vasoconstriction)
- Heart rate: ↑ 10-25 bpm
- Symptoms: None or mild lightheadedness (transient)

**References:**
- Brignole M et al. (2018) ESC Guidelines for syncope diagnosis and management. *Eur Heart J* 39:1883-1948
- Freeman R et al. (2011) Consensus statement on orthostatic hypotension. *J Neurol Sci* 305:1-10
- Sheldon RS et al. (2015) POTS diagnostic criteria. *Heart Rhythm* 12:e41-e63

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.autonomic.autonomic_nervous_system import (
    AutonomicNervousSystem,
    AutonomicParameters,
)
from src.validation.benchmarks import PhysiologicalBenchmarks

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 12)
print("✓ Imports successful")

## Part 1: Normal Orthostatic Response (Tilt Table Test)

The tilt table test is the gold standard for evaluating orthostatic intolerance. Patient is tilted to 60-80° for 10-45 minutes while monitoring BP and HR.

**Protocol:**
- 5-10 min supine baseline
- Rapid tilt to 70° (< 15 seconds)
- Maintain upright for 10-45 min or until symptoms
- Return to supine

In [ ]:
# Simulate normal tilt table test using the built-in method
ans_normal = AutonomicNervousSystem()

print("="*60)
print("TILT TABLE TEST - NORMAL RESPONSE")
print("="*60)
print("\nSimulating 70° tilt for 10 minutes...\n")

results_normal = ans_normal.simulate_orthostatic_stress(
    duration=15.0 * 60,  # 15 minutes total (seconds)
    tilt_time=5.0 * 60,  # Tilt at 5 minutes (seconds)
    pressure_drop=20.0,  # Initial pressure drop from venous pooling
    dt=0.01,
)

# Convert to numpy arrays and to minutes
times = np.array(results_normal['time']) / 60.0  # Convert to minutes
pressure = np.array(results_normal['pressure'])
hr = np.array(results_normal['heart_rate'])
vagal = np.array(results_normal['vagal_tone'])
sympathetic = np.array(results_normal['sympathetic_tone'])
svr = np.array(results_normal['svr_multiplier'])

# Create comprehensive visualization
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)

# Mark phases
tilt_time_min = 5.0
for ax in axes:
    ax.axvspan(0, tilt_time_min, alpha=0.1, color='lightblue', label='Supine')
    ax.axvspan(tilt_time_min, 15, alpha=0.1, color='lightyellow', label='Upright (70°)')
    ax.axvline(tilt_time_min, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Plot 1: Blood Pressure
axes[0].plot(times, pressure, 'b-', linewidth=2.5)
axes[0].axhline(93, color='gray', linestyle='--', alpha=0.5, label='Baseline MAP')
axes[0].axhline(73, color='red', linestyle='--', alpha=0.7, label='Hypotension threshold (Δ-20)')
axes[0].set_ylabel('Mean Arterial\nPressure (mmHg)', fontsize=12, fontweight='bold')
axes[0].set_title('Tilt Table Test - Normal Orthostatic Response', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='lower left')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([60, 110])

# Plot 2: Heart Rate
axes[1].plot(times, hr, 'r-', linewidth=2.5)
axes[1].axhline(75, color='gray', linestyle='--', alpha=0.5, label='Baseline HR')
axes[1].axhline(105, color='orange', linestyle='--', alpha=0.7, label='POTS threshold (+30 bpm)')
axes[1].set_ylabel('Heart Rate\n(bpm)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([60, 120])

# Plot 3: Systemic Vascular Resistance
axes[2].plot(times, svr, 'purple', linewidth=2.5)
axes[2].axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Baseline SVR')
axes[2].set_ylabel('SVR Multiplier\n(relative to baseline)', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0.8, 2.0])

# Plot 4: Autonomic Tone
axes[3].plot(times, vagal, 'g-', linewidth=2.5, label='Vagal (Parasympathetic)', alpha=0.8)
axes[3].plot(times, sympathetic, 'orange', linewidth=2.5, label='Sympathetic', alpha=0.8)
axes[3].set_ylabel('Autonomic Tone\n(0-1)', fontsize=12, fontweight='bold')
axes[3].legend(fontsize=10, loc='right')
axes[3].set_ylim([0, 1.0])
axes[3].grid(True, alpha=0.3)

# Plot 5: Cardiac Output Estimate
# CO = HR × SV, SV affected by preload (venous return) and contractility
baseline_sv = 70.0  # mL
# Estimate SV: reduced by venous pooling, increased by sympathetic contractility
sv_factor = np.where(times < tilt_time_min, 1.0, 0.7 + 0.3 * (sympathetic - 0.3) / 0.7)
sv = baseline_sv * sv_factor
co = (hr * sv) / 1000.0  # L/min

axes[4].plot(times, co, 'darkblue', linewidth=2.5)
axes[4].axhline(5.0, color='gray', linestyle='--', alpha=0.5, label='Baseline CO')
axes[4].set_xlabel('Time (minutes)', fontsize=12, fontweight='bold')
axes[4].set_ylabel('Cardiac Output\n(L/min)', fontsize=12, fontweight='bold')
axes[4].legend(fontsize=10)
axes[4].grid(True, alpha=0.3)
axes[4].set_ylim([3, 7])

plt.tight_layout()
plt.show()

# Calculate orthostatic metrics
baseline_idx = int(4.9 * 60 / 0.01)  # Just before tilt
upright_idx = int(5.5 * 60 / 0.01)   # 30 sec after tilt

baseline_pressure = pressure[baseline_idx]
baseline_hr_val = hr[baseline_idx]
upright_pressure = pressure[upright_idx]
upright_hr_val = hr[upright_idx]

delta_pressure = upright_pressure - baseline_pressure
delta_hr = upright_hr_val - baseline_hr_val

print("\n" + "="*60)
print("ORTHOSTATIC VITAL SIGNS")
print("="*60)
print(f"\nBaseline (Supine):")
print(f"  Blood Pressure: {baseline_pressure:.0f} mmHg")
print(f"  Heart Rate: {baseline_hr_val:.0f} bpm")

print(f"\nUpright (30 sec after tilt):")
print(f"  Blood Pressure: {upright_pressure:.0f} mmHg (Δ {delta_pressure:+.0f})")
print(f"  Heart Rate: {upright_hr_val:.0f} bpm (Δ {delta_hr:+.0f})")

print(f"\nInterpretation:")
if abs(delta_pressure) < 20 and delta_hr < 30:
    print(f"  ✓ Normal orthostatic response")
    print(f"  - Adequate baroreflex compensation")
    print(f"  - BP maintained within normal range")
    print(f"  - Appropriate HR increase")
    print(f"  - SVR increased by {(svr[upright_idx]-1)*100:.0f}% (vasoconstriction)")
elif delta_pressure < -20:
    print(f"  ⚠️  Orthostatic hypotension (OH)")
elif delta_hr > 30:
    print(f"  ⚠️  Excessive tachycardia - consider POTS")

print("\n" + "="*60)

## Part 2: Orthostatic Hypotension

**Definition (Consensus):**
- Sustained reduction in systolic BP ≥20 mmHg OR diastolic BP ≥10 mmHg within 3 minutes of standing

**Subtypes:**
1. **Initial OH**: Transient BP drop <30 sec (normal in young adults)
2. **Classic OH**: Sustained BP drop within 3 min
3. **Delayed OH**: BP drop after 3+ min of standing

**Causes:**
- Autonomic failure (diabetes, Parkinson's, MSA)
- Hypovolemia (dehydration, hemorrhage)
- Medications (alpha-blockers, diuretics, vasodilators)
- Deconditioning
- Aging

In [ ]:
from src.autonomic.baroreflex import BaroreflexParameters

# Simulate orthostatic hypotension (impaired autonomic compensation)
oh_baroreflex = BaroreflexParameters(
    max_firing_rate=60.0,    # Reduced from 100 (impaired baroreceptors)
    sigmoid_slope=0.05,      # Reduced from 0.1 (blunted sensitivity)
    vagal_gain=0.3,          # Reduced from 0.6
    sympathetic_gain=0.3,    # Reduced from 0.6
)

oh_params = AutonomicParameters(
    baseline_vagal_tone=0.5,
    baseline_sympathetic_tone=0.4,
    max_sympathetic_vasoconstriction=1.5,  # Reduced from 3.0 (impaired vasoconstriction)
)

ans_oh = AutonomicNervousSystem(
    params=oh_params,
    baroreflex_params=oh_baroreflex
)

print("="*60)
print("ORTHOSTATIC HYPOTENSION SIMULATION")
print("="*60)
print("\nSimulating patient with autonomic failure...\n")

results_oh = ans_oh.simulate_orthostatic_stress(
    duration=15.0 * 60,
    tilt_time=5.0 * 60,
    pressure_drop=30.0,  # Larger drop due to impaired compensation
    dt=0.01,
)

# Extract data
times_oh = np.array(results_oh['time']) / 60.0
pressure_oh = np.array(results_oh['pressure'])
hr_oh = np.array(results_oh['heart_rate'])
svr_oh = np.array(results_oh['svr_multiplier'])

# Compare normal vs OH
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)

for ax in axes:
    ax.axvspan(0, 5, alpha=0.1, color='lightblue')
    ax.axvspan(5, 15, alpha=0.1, color='lightyellow')
    ax.axvline(5, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Plot 1: Blood Pressure Comparison
axes[0].plot(times, pressure, 'b-', linewidth=2.5, label='Normal', alpha=0.8)
axes[0].plot(times_oh, pressure_oh, 'r--', linewidth=2.5, label='Orthostatic Hypotension', alpha=0.8)
axes[0].axhline(93, color='gray', linestyle=':', alpha=0.5, label='Baseline')
axes[0].axhline(73, color='red', linestyle=':', alpha=0.7, label='OH threshold (-20 mmHg)')
axes[0].axhline(60, color='darkred', linestyle=':', alpha=0.7, label='Severe hypotension')
axes[0].set_ylabel('Mean Arterial Pressure (mmHg)', fontsize=12, fontweight='bold')
axes[0].set_title('Orthostatic Hypotension vs Normal Response', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='lower left')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([50, 100])

# Plot 2: Heart Rate Comparison
axes[1].plot(times, hr, 'b-', linewidth=2.5, label='Normal', alpha=0.8)
axes[1].plot(times_oh, hr_oh, 'r--', linewidth=2.5, label='Orthostatic Hypotension', alpha=0.8)
axes[1].axhline(75, color='gray', linestyle=':', alpha=0.5)
axes[1].set_ylabel('Heart Rate (bpm)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(True, alpha=0.3)

# Plot 3: SVR Comparison
axes[2].plot(times, svr, 'b-', linewidth=2.5, label='Normal (vasoconstriction)', alpha=0.8)
axes[2].plot(times_oh, svr_oh, 'r--', linewidth=2.5, label='OH (impaired vasoconstriction)', alpha=0.8)
axes[2].axhline(1.0, color='gray', linestyle=':', alpha=0.5)
axes[2].set_xlabel('Time (minutes)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('SVR Multiplier', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0.8, 2.2])

plt.tight_layout()
plt.show()

# Calculate metrics
baseline_idx_oh = int(4.9 * 60 / 0.01)
upright_idx_oh = int(5.5 * 60 / 0.01)

delta_pressure_oh = pressure_oh[upright_idx_oh] - pressure_oh[baseline_idx_oh]
delta_hr_oh = hr_oh[upright_idx_oh] - hr_oh[baseline_idx_oh]

print("\n" + "="*60)
print("ORTHOSTATIC HYPOTENSION ASSESSMENT")
print("="*60)

print(f"\nNormal Individual:")
print(f"  ΔBP: {delta_pressure:+.0f} mmHg")
print(f"  ΔHR: {delta_hr:+.0f} bpm")
print(f"  ΔSVR: +{(svr[upright_idx]-1)*100:.0f}% (adequate compensation)")
print(f"  Result: Normal")

print(f"\nOrthostatic Hypotension Patient:")
print(f"  ΔBP: {delta_pressure_oh:+.0f} mmHg {'⚠️  ABNORMAL (meets OH criteria)' if delta_pressure_oh < -20 else ''}")
print(f"  ΔHR: {delta_hr_oh:+.0f} bpm (inadequate compensatory tachycardia)")
print(f"  ΔSVR: +{(svr_oh[upright_idx_oh]-1)*100:.0f}% (impaired vasoconstriction)")
print(f"  Lowest BP: {min(pressure_oh):.0f} mmHg")

print(f"\nClinical Significance:")
if delta_pressure_oh < -20:
    print(f"  - Meets criteria for orthostatic hypotension")
    print(f"  - Impaired sympathetic vasoconstriction")
    print(f"  - High risk for falls and syncope")
    print(f"  - Differential diagnosis:")
    print(f"    • Autonomic failure (diabetes, Parkinson's, MSA)")
    print(f"    • Medications (alpha-blockers, diuretics)")
    print(f"    • Hypovolemia")
    print(f"    • Deconditioning")

print(f"\nManagement Recommendations:")
print(f"  1. Non-pharmacological:")
print(f"     - Increase fluid/salt intake (8-10 g NaCl/day)")
print(f"     - Compression stockings (30-40 mmHg)")
print(f"     - Physical countermaneuvers (leg crossing, squatting)")
print(f"     - Elevate head of bed 10-30°")
print(f"  2. Pharmacological (if non-pharm insufficient):")
print(f"     - Fludrocortisone 0.1-0.4 mg/day")
print(f"     - Midodrine 5-10 mg TID")
print(f"     - Droxidopa (for neurogenic OH)")
print(f"  3. Review and adjust medications")

print("\n" + "="*60)

## Part 3: Postural Orthostatic Tachycardia Syndrome (POTS)

**Definition:**
- Heart rate increase ≥30 bpm (or to ≥120 bpm) within 10 min of standing
- In absence of orthostatic hypotension (BP drop <20 mmHg systolic)
- Accompanied by symptoms: lightheadedness, palpitations, tremulousness, weakness

**Prevalence:**
- 0.2% of population
- 80% female, typical onset 15-50 years
- Post-viral trigger common (including post-COVID "long COVID")

**Proposed Mechanisms:**
1. **Neuropathic**: Peripheral denervation → venous pooling
2. **Hyperadrenergic**: Excessive sympathetic activation
3. **Hypovolemic**: Reduced plasma volume
4. **Deconditioning**: Reduced cardiac/skeletal muscle tone

In [ ]:
# Simulate POTS (hyperadrenergic subtype)
pots_params = AutonomicParameters(
    baseline_vagal_tone=0.5,
    baseline_sympathetic_tone=0.6,  # Elevated baseline
    max_sympathetic_hr_effect=80.0,  # Increased from 60 (excessive tachycardia)
    sympathetic_time_constant=1.0,  # Faster from 2.0 (exaggerated response)
    max_sympathetic_vasoconstriction=2.0,  # Somewhat reduced (peripheral denervation)
)

# Normal baroreflex (POTS is not primarily baroreflex dysfunction)
ans_pots = AutonomicNervousSystem(params=pots_params)

print("="*60)
print("POSTURAL ORTHOSTATIC TACHYCARDIA SYNDROME (POTS)")
print("="*60)
print("\nSimulating hyperadrenergic POTS patient...\n")

results_pots = ans_pots.simulate_orthostatic_stress(
    duration=15.0 * 60,
    tilt_time=5.0 * 60,
    pressure_drop=25.0,  # Moderate drop (not severe OH)
    dt=0.01,
)

times_pots = np.array(results_pots['time']) / 60.0
pressure_pots = np.array(results_pots['pressure'])
hr_pots = np.array(results_pots['heart_rate'])
vagal_pots = np.array(results_pots['vagal_tone'])
symp_pots = np.array(results_pots['sympathetic_tone'])

# Compare all three conditions
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

for ax in axes:
    ax.axvspan(0, 5, alpha=0.1, color='lightblue')
    ax.axvspan(5, 15, alpha=0.1, color='lightyellow')
    ax.axvline(5, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Tilt')

# Plot 1: Blood Pressure (all three)
axes[0].plot(times, pressure, 'g-', linewidth=2.5, label='Normal', alpha=0.8)
axes[0].plot(times_oh, pressure_oh, 'r--', linewidth=2.5, label='Orthostatic Hypotension', alpha=0.8)
axes[0].plot(times_pots, pressure_pots, 'b-.', linewidth=2.5, label='POTS', alpha=0.8)
axes[0].axhline(93, color='gray', linestyle=':', alpha=0.5)
axes[0].axhline(73, color='red', linestyle=':', alpha=0.5, label='OH threshold')
axes[0].set_ylabel('Mean Arterial Pressure (mmHg)', fontsize=12, fontweight='bold')
axes[0].set_title('Orthostatic Intolerance - Differential Diagnosis', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='lower left', ncol=2)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([50, 105])

# Plot 2: Heart Rate (all three)
axes[1].plot(times, hr, 'g-', linewidth=2.5, label='Normal', alpha=0.8)
axes[1].plot(times_oh, hr_oh, 'r--', linewidth=2.5, label='Orthostatic Hypotension', alpha=0.8)
axes[1].plot(times_pots, hr_pots, 'b-.', linewidth=2.5, label='POTS', alpha=0.8)
axes[1].axhline(75, color='gray', linestyle=':', alpha=0.5, label='Baseline')
axes[1].axhline(105, color='orange', linestyle=':', alpha=0.7, label='POTS threshold (+30 bpm)')
axes[1].axhline(120, color='red', linestyle=':', alpha=0.7, label='Absolute POTS threshold (120 bpm)')
axes[1].set_xlabel('Time (minutes)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Heart Rate (bpm)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left', ncol=2)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([60, 140])

plt.tight_layout()
plt.show()

# Diagnostic criteria evaluation
baseline_idx_pots = int(4.9 * 60 / 0.01)
upright_idx_pots = int(5.5 * 60 / 0.01)

delta_pressure_pots = pressure_pots[upright_idx_pots] - pressure_pots[baseline_idx_pots]
delta_hr_pots = hr_pots[upright_idx_pots] - hr_pots[baseline_idx_pots]

print("\n" + "="*60)
print("DIFFERENTIAL DIAGNOSIS OF ORTHOSTATIC INTOLERANCE")
print("="*60)

print(f"\n{'Condition':<25} {'ΔBP (mmHg)':<15} {'ΔHR (bpm)':<15} {'Diagnosis'}")
print("-" * 80)

print(f"{'Normal':<25} {delta_pressure:+6.0f} {'':>8} {delta_hr:+6.0f} {'':>8} {'✓ Normal'}")

oh_diagnosis = "OH" if delta_pressure_oh < -20 else "Borderline"
print(f"{'Autonomic Failure':<25} {delta_pressure_oh:+6.0f} {'':>8} {delta_hr_oh:+6.0f} {'':>8} {oh_diagnosis}")

pots_diagnosis = "POTS" if (delta_hr_pots >= 30 and delta_pressure_pots >= -20) else "Not POTS"
print(f"{'POTS':<25} {delta_pressure_pots:+6.0f} {'':>8} {delta_hr_pots:+6.0f} {'':>8} {pots_diagnosis}")

print(f"\nDiagnostic Criteria:")
print(f"  Orthostatic Hypotension: SBP ↓≥20 or DBP ↓≥10 mmHg")
print(f"  POTS: HR ↑≥30 bpm (or to ≥120 bpm) WITHOUT OH")

if pots_diagnosis == "POTS":
    print(f"\nPOTS Management:")
    print(f"  1. Lifestyle modifications:")
    print(f"     - Increase fluid intake (2-3 L/day)")
    print(f"     - Increase salt intake (6-10 g/day)")
    print(f"     - Avoid triggers (heat, prolonged standing, large meals)")
    print(f"     - Compression garments (waist-high, 30-40 mmHg)")
    print(f"  2. Exercise therapy:")
    print(f"     - Gradual reconditioning (start supine/recumbent)")
    print(f"     - Swimming, rowing, recumbent cycling")
    print(f"     - Progress to upright exercise over 3-6 months")
    print(f"  3. Pharmacotherapy (if lifestyle insufficient):")
    print(f"     - Propranolol 10-20 mg TID (blocks tachycardia)")
    print(f"     - Fludrocortisone 0.1-0.2 mg/day (volume expansion)")
    print(f"     - Midodrine 5-10 mg TID (vasoconstriction)")
    print(f"     - Ivabradine 5-7.5 mg BID (selective HR reduction)")
    print(f"  4. Symptom management:")
    print(f"     - Pyridostigmine (if neuropathic subtype)")
    print(f"     - Cognitive behavioral therapy (if significant anxiety component)")

print("\n" + "="*60)
print("REFERENCES:")
print("  - Sheldon RS et al. (2015) Heart Rhythm 12:e41-e63")
print("  - Raj SR (2013) Circulation 127:2336-2342")
print("  - Bryarly M et al. (2019) J Am Heart Assoc 8:e013046")
print("="*60)

## Summary and Clinical Pearls

### Key Concepts

**1. Normal Orthostatic Physiology:**
- Gravitational pooling of 500-1000 mL blood upon standing
- Immediate baroreflex activation (within 1-2 cardiac cycles)
- Coordinated response: vagal withdrawal + sympathetic activation
- Normal vital sign changes: HR ↑10-25 bpm, SBP ↓<20 mmHg, DBP ↑5-10 mmHg

**2. Orthostatic Hypotension (OH):**
- Definition: SBP ↓≥20 or DBP ↓≥10 mmHg within 3 min of standing
- Mechanism: Impaired sympathetic vasoconstriction and/or hypovolemia
- Symptoms: Lightheadedness, visual blurring, weakness, syncope
- High risk for falls, fractures, reduced quality of life
- Treatment: Volume expansion, compression, medications (fludrocortisone, midodrine)

**3. Postural Orthostatic Tachycardia Syndrome (POTS):**
- Definition: HR ↑≥30 bpm (or to ≥120 bpm) within 10 min WITHOUT OH
- Predominantly young females (80%)
- Subtypes: Neuropathic, hyperadrenergic, hypovolemic
- Debilitating symptoms but NOT life-threatening
- Treatment: Reconditioning, volume expansion, beta-blockers, ivabradine

**4. Differential Diagnosis:**
| Condition | ΔBP | ΔHR | Key Features |
|-----------|-----|-----|-------------|
| **Normal** | <20 ↓ | +10-25 | Asymptomatic, good compensation |
| **OH** | ≥20 ↓ | Variable | Impaired vasoconstriction, symptoms |
| **POTS** | <20 ↓ | ≥30 ↑ | Excessive tachycardia, debilitating symptoms |
| **Vasovagal** | Severe ↓ | Severe ↓ (bradycardia) | Prodrome, pallor, nausea, recovery rapid |

### Clinical Approach to Syncope/Orthostatic Intolerance

**1. History:**
- Circumstances (standing, exertion, emotional stress)
- Prodrome (lightheadedness, nausea, visual changes)
- Witness account (duration, color, movements)
- Recovery (immediate vs prolonged confusion)
- Medications (especially new or increased doses)

**2. Physical Examination:**
- **Orthostatic vital signs** (supine, then at 1 and 3 min standing)
  - Measure BP and HR at each time point
  - Ask about symptoms at each position
- Cardiac exam (murmurs suggesting AS or HCM)
- Neurological exam (autonomic testing)

**3. Investigations:**
- **Initial:** ECG, CBC, electrolytes, glucose
- **If recurrent/unexplained:**
  - Tilt table test (gold standard)
  - Holter monitor (if palpitations)
  - Echocardiogram (if cardiac suspected)
  - Autonomic testing (QSART, Valsalva, etc.)

**4. Risk Stratification:**
- **High risk** (admission/urgent workup):
  - Age >60 with no obvious benign cause
  - Exertional syncope
  - Syncope while supine
  - Palpitations before syncope
  - Family history of sudden cardiac death
  - Structural heart disease or abnormal ECG
- **Low risk** (outpatient evaluation):
  - Young, healthy
  - Clear vasovagal trigger
  - Normal cardiac exam and ECG
  - Rapid, complete recovery

### Take-Home Messages

1. **Orthostatic vital signs are mandatory** in syncope evaluation
2. **POTS is NOT orthostatic hypotension** - different pathophysiology and treatment
3. **Most syncope is benign** (vasovagal), but cardiac causes are life-threatening
4. **Non-pharmacological measures** are first-line for OH and POTS
5. **Exercise reconditioning** is highly effective in POTS (often curative)
6. **Medication review** is critical - many drugs cause/worsen orthostatic intolerance

### References

- Brignole M et al. (2018) ESC Guidelines for syncope. *Eur Heart J* 39:1883-1948
- Freeman R et al. (2011) Consensus on orthostatic hypotension. *J Neurol Sci* 305:1-10
- Sheldon RS et al. (2015) POTS Statement. *Heart Rhythm* 12:e41-e63
- Raj SR (2013) POTS. *Circulation* 127:2336-2342

---
© 2025 Multi-Heart-Model Project | MIT License